## 1. Load libraries

In [50]:
import gc
import json
import os, h5py
import math
import multiprocessing
import numpy as np
import pandas as pd
import torch
import importlib
import logging
from pathlib import Path
from sklearn.model_selection import GroupKFold, GroupShuffleSplit
from sklearn.utils import resample
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor
import multiprocessing as mp
mp.set_start_method('spawn', force=True)

# Pycox and PyTorch tuples for survival analysis
import torchtuples as tt
import pycox
from pycox.preprocessing.label_transforms import LabTransDiscreteTime
from pycox.models import CoxPH, DeepHit
from pycox.evaluation import EvalSurv
from lifelines import KaplanMeierFitter

# Ray for hyperparameter tuning and distributed processing
import ray
from ray import tune
from ray.tune import CLIReporter
from ray.tune.search.bayesopt import BayesOptSearch
from ray.tune.search.optuna import OptunaSearch
from ray.tune.search import ConcurrencyLimiter
from ray.tune.schedulers import ASHAScheduler, PopulationBasedTraining
from ray.air import session
import ray.cloudpickle as pickle

# Custom modules for data handling, balancing, training, evaluation, and model architectures
import dataloader2
import databalancer2
import datatrainer2
import modeleval
import netweaver2
import avg_ensemble

# Reload custom modules to ensure latest changes are available
importlib.reload(dataloader2)
importlib.reload(databalancer2)
importlib.reload(datatrainer2)
importlib.reload(modeleval)
importlib.reload(netweaver2)
importlib.reload(avg_ensemble)

# Import specific functions from custom modules to keep code clean and readable
from netweaver2 import (
    lstm_net_init, DHANNWrapper, LSTMWrapper, generalized_ann_net_init
)
from dataloader2 import (
    load_and_transform_data, preprocess_data #stack_sequences, dh_dataset_loader
)
from databalancer2 import (
    define_medoid_general, df_event_focus, rebalance_data, underbalance_data_general, medoid_cluster, 
    dh_rebalance_data
)
from datatrainer2 import (
    recursive_clustering, prepare_training_data, 
    prepare_validation_data, lstm_training
)
from modeleval import (
    dh_test_model, nam_dagostino_chi2, get_baseline_hazard_at_timepoints, combined_test_model
)
from avg_ensemble import (
    load_bootstrap_predictions, generate_combinations, get_cif_for_combination, ensemble_cif_arrays, save_metrics_to_individual_json,
    get_processed_combinations, process_combination, process_combination_worker
)

import psutil
torch.cuda.empty_cache()
gc.collect()

111

## 2. Define Constants and Load Datasets

In [51]:

RANDOM_SEED = 12345
N_SPLIT = 2
FEATURE_COLS = ['gender', 'dm', 'ht', 'sprint', 'a1c', 'po4', 'UACR_mg_g', 'Cr', 'age', 'alb', 'ca', 'hb', 'hco3']
DURATION_COL = 'date_from_sub_60'
EVENT_COL = 'endpoint'
CLUSTER_COL = 'key'
TIME_GRID = np.array([i * 365 for i in range(6)])

# Define Feature Groups
CAT_FEATURES = ['gender', 'dm', 'ht', 'sprint']
LOG_FEATURES = ['a1c', 'po4', 'UACR_mg_g', 'Cr']
STANDARD_FEATURES = ['age', 'alb', 'ca', 'hb', 'hco3']
PASSTHROUGH_FEATURES = ['key', 'date_from_sub_60', 'endpoint']

# Load and Transform Data
BASE_FILENAME = '/mnt/d/pydatascience/g3_regress/data/X/X_20240628'
X_train_transformed, X_test_transformed = load_and_transform_data(
    BASE_FILENAME, CAT_FEATURES, LOG_FEATURES, STANDARD_FEATURES, PASSTHROUGH_FEATURES
)

2025-04-26 14:33:41,173 - INFO - Transforming training data...
2025-04-26 14:33:56,949 - INFO - Transforming test data...
/home/goma/miniconda3/envs/ai_dev/lib/python3.11/site-packages/sklearn/pipeline.py:62: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.
  warnings.warn(


In [148]:
X_train_transformed

,gender,dm,ht,sprint,a1c,po4,UACR_mg_g,Cr,age,alb,ca,hb,hco3,key,date_from_sub_60,endpoint,eGFRcr,A_class,G_class
0,0.0,0.0,1.0,0.0,0.176112,0.652178,0.859990,0.530212,0.522223,0.473685,0.354245,0.396136,0.388146,1625799,1287.0,0.0,19.165777,A3,G4
1,0.0,0.0,0.0,0.0,0.178607,0.629492,0.777839,0.250621,0.511112,0.543860,0.362778,0.420290,0.409851,1635448,585.0,0.0,97.303617,A2,G1
2,1.0,0.0,0.0,0.0,0.203166,0.640226,0.805509,0.268433,0.600000,0.491229,0.338385,0.603865,0.625239,2176480,1190.0,0.0,86.787429,A3,G2
3,0.0,0.0,0.0,1.0,0.170315,0.602436,0.697881,0.206173,0.366667,0.578948,0.403973,0.521740,0.447419,6718845,1458.0,0.0,112.740038,A1,G1
4,1.0,0.0,1.0,0.0,0.165390,0.552711,0.670223,0.315965,0.555556,0.578948,0.412006,0.676329,0.516253,2838012,223.0,0.0,67.100755,A1,G2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
396419,1.0,0.0,0.0,0.0,0.167192,0.606490,0.797260,0.302017,0.811111,0.456141,0.373195,0.555556,0.483748,4968538,90.0,2.0,63.17716,A3,G2
396420,1.0,0.0,1.0,1.0,0.178607,0.569661,0.798349,0.323523,0.555556,0.526316,0.398912,0.690822,0.445553,5297180,81.0,0.0,64.156811,A3,G2
396421,1.0,0.0,1.0,1.0,0.187056,0.629492,0.818211,0.471831,0.533334,0.561404,0.392203,0.584541,0.363289,2080829,390.0,0.0,26.935532,A3,G4
396422,0.0,0.0,1.0,1.0,0.126422,0.545069,0.704616,0.277948,0.688889,0.350878,0.354687,0.396136,0.363289,7211405,1270.0,0.0,78.040379,A2,G2


## 3. Load test set bootstrap data

In [188]:
# List of CIF labels
cif_array_labels = [
    "deepsurv_ann_clustering",
    "deepsurv_ann_enn",
    "deepsurv_ann_tomek",
    "deepsurv_lstm_clustering",
    "deepsurv_lstm_NearMiss",
    "deephit_ann_clustering",
    "deephit_ann_NearMiss",
    "deephit_lstm_clustering",
    "deephit_lstm_NearMiss",
]

# Directory containing the bootstrap HDF5 files
directory_path = '/mnt/d/PYDataScience/g3_regress/data/results/bootstrap_predictions/20250424_test/'
output_dir = "/mnt/d/pydatascience/g3_regress/data/results/risk_by_ag_stage/"

# Load all bootstrap predictions
bootstrap_data = load_bootstrap_predictions(directory_path)
all_combinations = generate_combinations(cif_array_labels)
predictions = bootstrap_data["predictions"]
durations = bootstrap_data["durations"]
events = bootstrap_data["events"]
a_class = bootstrap_data["a_class"]
g_class = bootstrap_data["g_class"]
combo = [max(all_combinations, key=len)]

Loading /mnt/d/PYDataScience/g3_regress/data/results/bootstrap_predictions/20250424_test/bootstrap_iteration_1.h5...
Loading /mnt/d/PYDataScience/g3_regress/data/results/bootstrap_predictions/20250424_test/bootstrap_iteration_10.h5...
Loading /mnt/d/PYDataScience/g3_regress/data/results/bootstrap_predictions/20250424_test/bootstrap_iteration_100.h5...
Loading /mnt/d/PYDataScience/g3_regress/data/results/bootstrap_predictions/20250424_test/bootstrap_iteration_101.h5...
Loading /mnt/d/PYDataScience/g3_regress/data/results/bootstrap_predictions/20250424_test/bootstrap_iteration_102.h5...
Loading /mnt/d/PYDataScience/g3_regress/data/results/bootstrap_predictions/20250424_test/bootstrap_iteration_103.h5...
Loading /mnt/d/PYDataScience/g3_regress/data/results/bootstrap_predictions/20250424_test/bootstrap_iteration_104.h5...
Loading /mnt/d/PYDataScience/g3_regress/data/results/bootstrap_predictions/20250424_test/bootstrap_iteration_105.h5...
Loading /mnt/d/PYDataScience/g3_regress/data/result

## 4. Create result dataframe for test set

In [189]:
data = {
    'event': [],
    'Time Point': [],
    'A Class': [],
    'G Class': [],
    'Predicted': [],
    'Observed': []
}

for i, itr_predictions in enumerate((bootstrap_data['predictions'].values())):
    # itr_predictions = bootstrap_data['predictions'][i]
    durations = bootstrap_data['durations'][i]
    events = bootstrap_data['events'][i]
    a_class = bootstrap_data['a_class'][i]
    g_class = bootstrap_data['g_class'][i]

    combo = tuple(itr_predictions.keys())
    cif_array = get_cif_for_combination(itr_predictions, combo)
    ensembled_cif = ensemble_cif_arrays(cif_array)

    for e in range(ensembled_cif.shape[0]):
        for t in range(1, ensembled_cif.shape[1]):  # time points 1y to 5y
            probs = ensembled_cif[e][t]
            for ac in np.unique(a_class):
                for gc in np.unique(g_class):
                    mask = (a_class == ac) & (g_class == gc)
                    if not np.any(mask):
                        continue

                    kmf = KaplanMeierFitter()
                    kmf.fit(durations[mask], events[mask] == (e + 1))

                    if t not in kmf.cumulative_density_.index:
                        closest = kmf.cumulative_density_.index.get_indexer([t], method='nearest')[0]
                        obs = kmf.cumulative_density_.iloc[closest].values[0]
                    else:
                        obs = kmf.cumulative_density_.loc[t].values[0]

                    pred = np.mean(probs[mask])

                    data['event'].append(e + 1)
                    data['Time Point'].append(t)
                    data['A Class'].append(ac)
                    data['G Class'].append(gc)
                    data['Predicted'].append(pred)
                    data['Observed'].append(obs)

result_df = pd.DataFrame(data)

## 5. Plot predict and observe risk calibration in test set

In [139]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.transforms import Bbox
import numpy as np

def plot_event_risk(result_df, event_num, save_path=None):
    fig = plt.figure(figsize=(20, 16))
    gs = gridspec.GridSpec(4, 3, figure=fig)
    a_labels = ['A1','A2','A3']
    g_labels = ['G3a','G3b','G4','G5']

    axes = [[None]*3 for _ in range(4)]
    for i, g in enumerate(g_labels):
        for j, a in enumerate(a_labels):
            ax = fig.add_subplot(gs[i, j])
            axes[i][j] = ax

            means = ( result_df
                      [(result_df['A Class']==a)&
                       (result_df['G Class']==g)&
                       (result_df['event']==event_num)]
                      .groupby('Time Point')[['Predicted','Observed']]
                      .mean().reindex([1,2,3,4,5]) )

            if not means.empty:
                x = np.arange(5)
                width = 0.35

                # **ONLY** label on the first subplot so we don't duplicate in legend
                if i==0 and j==0:
                    ax.bar(
                        x-width/2,
                        means['Predicted'],
                        width=width,
                        color='skyblue',
                        label='Predicted'
                    )
                    ax.bar(
                        x+width/2,
                        means['Observed'],
                        width=width,
                        color='orange',
                        label='Observed'
                    )
                else:
                    ax.bar(x-width/2, means['Predicted'], width=width, color='skyblue')
                    ax.bar(x+width/2, means['Observed'], width=width, color='orange')

            if j==0:
                ax.set_ylabel("Risk")
            else:
                ax.set_yticklabels([])

            if i==3:
                ax.set_xticks(x)
                ax.set_xticklabels(['1y','2y','3y','4y','5y'])
            else:
                ax.set_xticks([])

            ax.set_ylim(0,1)
            ax.set_title("")

    # column headers
    for j, lbl in enumerate(a_labels):
        top = axes[0][j].get_position()
        fig.text(
            top.x0+top.width/2, top.y1+0.01,
            lbl, ha='center', va='bottom', fontsize=16, fontweight='bold'
        )

    # row headers
    for i, lbl in enumerate(g_labels):
        boxes = [axes[i][j].get_position() for j in range(3)]
        row_bbox = Bbox.union(boxes)
        fig.text(
            row_bbox.x0 - 0.05,
            row_bbox.y0+row_bbox.height/2 - (i * 0.025),
            lbl, ha='right', va='center', fontsize=16, fontweight='bold'
        )

    # now grab the handles and labels from the first axis
    handles, labels = axes[0][0].get_legend_handles_labels()
    fig.legend(handles, labels, loc='upper center', ncol=2, fontsize='large')

    plt.tight_layout(rect=[0.08,0,0.95,0.88])
    plt.suptitle(f"Event {event_num}: Mean Predicted vs Mean Observed Risk", fontsize=22, y=0.96)
    
    if save_path:
        # make dirs if needed
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        fig.savefig(save_path, dpi=1000, bbox_inches='tight')
        plt.close(fig)
    else:
        plt.show()


In [140]:
plot_event_risk(
    result_df,
    event_num=1,
    save_path='/mnt/d/pydatascience/g3_regress/doc/event_1_by_ag_stage.png'
)


In [141]:
plot_event_risk(
    result_df,
    event_num=2,
    save_path='/mnt/d/pydatascience/g3_regress/doc/event_2_by_ag_stage.png'
)

## 6. Plot observed risk of Event 1 and 2 in test set

In [152]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.transforms import Bbox

def plot_observed_events(result_df, save_path=None):
    fig = plt.figure(figsize=(20, 16))
    gs  = gridspec.GridSpec(4, 3, figure=fig)
    a_labels = ['A1','A2','A3']
    g_labels = ['G3a','G3b','G4','G5']

    # store axes for later labelling
    axes = [[None]*3 for _ in range(4)]

    for i, g in enumerate(g_labels):
        for j, a in enumerate(a_labels):
            ax = fig.add_subplot(gs[i, j])
            axes[i][j] = ax

            # compute mean observed for each event
            means1 = ( result_df
                       [(result_df['A Class']==a)&
                        (result_df['G Class']==g)&
                        (result_df['event']==1)]
                       .groupby('Time Point')['Observed']
                       .mean().reindex([1,2,3,4,5], fill_value=0) )

            means2 = ( result_df
                       [(result_df['A Class']==a)&
                        (result_df['G Class']==g)&
                        (result_df['event']==2)]
                       .groupby('Time Point')['Observed']
                       .mean().reindex([1,2,3,4,5], fill_value=0) )

            x     = np.arange(5)  # 0..4
            width = 0.35

            # only label once for the legend
            if i==0 and j==0:
                ax.bar(x - width/2, means1.values, width,
                       color='skyblue', label='Event 1 Obs')
                ax.bar(x + width/2, means2.values, width,
                       color='orange',   label='Event 2 Obs')
            else:
                ax.bar(x - width/2, means1.values, width, color='skyblue')
                ax.bar(x + width/2, means2.values, width, color='orange')

            # y‐axis only on leftmost column
            if j==0:
                ax.set_ylabel("Risk")
            else:
                ax.set_yticklabels([])

            # x‐ticks only on bottom row
            if i==3:
                ax.set_xticks(x)
                ax.set_xticklabels(['1y','2y','3y','4y','5y'])
            else:
                ax.set_xticks([])

            ax.set_ylim(0, 1)

    # column headers centered
    for j, lbl in enumerate(a_labels):
        top = axes[0][j].get_position()
        fig.text(top.x0 + top.width/2, top.y1 + 0.02,
                 lbl, ha='center', va='bottom',
                 fontsize=16, fontweight='bold')

    # row headers centered alongside full row
    for i, lbl in enumerate(g_labels):
        row_boxes = [axes[i][j].get_position() for j in range(3)]
        row_bbox  = Bbox.union(row_boxes)
        y = row_bbox.y0 + row_bbox.height/2
        fig.text(0.02, y, lbl,
                 ha='right', va='center',
                 fontsize=16, fontweight='bold')

    # global legend
    handles, labels = axes[0][0].get_legend_handles_labels()
    fig.legend(handles, labels,
               loc='upper center', ncol=2, fontsize='large')

    plt.tight_layout(rect=[0.08, 0, 0.95, 0.88])
    plt.suptitle("Observed Risk: Event 1 vs Event 2 by A/G Class",
                 fontsize=22, y=0.96)
    if save_path:
        # make dirs if needed
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        fig.savefig(save_path, dpi=1000, bbox_inches='tight')
        plt.close(fig)
    else:
        plt.show()


In [ ]:
plot_observed_events(result_df, 
                     save_path='/mnt/d/pydatascience/g3_regress/doc/observed_events_by_ag_stage.png'
                    )


## 8. Load training set prediction

In [191]:
import glob
def load_all_bootstrap(root_dir):
    """
    Walk all sub‐dirs keys_001, keys_002, … under root_dir,
    load their bootstrap_iteration_*.h5 files, and merge into one dict.
    """
    merged = {
        "predictions": {},
        "durations": [],
        "events": [],
        "a_class": [],
        "g_class": []
    }

    # find each chunk folder
    chunk_dirs = sorted(glob.glob(os.path.join(root_dir, "keys_*")))

    for chunk in chunk_dirs:
        # load this chunk’s results
        chunk_data = load_bootstrap_predictions(chunk)

        # for each iteration in this chunk, prefix its key to avoid collisions
        for iter_idx, preds in chunk_data["predictions"].items():
            merge_key = f"{os.path.basename(chunk)}_{iter_idx}"
            merged["predictions"][merge_key] = preds

        # extend the lists
        merged["durations"].extend(chunk_data["durations"])
        merged["events"].extend(chunk_data["events"])
        merged["a_class"].extend(chunk_data["a_class"])
        merged["g_class"].extend(chunk_data["g_class"])

    return merged

root = "/mnt/d/pydatascience/g3_regress/data/results/bootstrap_predictions/20250427_train"
bootstrap_data = load_all_bootstrap(root)

flat_preds = {}
for iter_key, pred_dict in bootstrap_data['predictions'].items():
    for model_name, cif_arr in pred_dict.items():
        flat_preds.setdefault(model_name, []).append(cif_arr)

bootstrap_data['predictions'] = flat_preds
for model_name, arr_list in bootstrap_data['predictions'].items():
    combined = np.concatenate(arr_list, axis=-1)
    bootstrap_data['predictions'][model_name] = combined

bootstrap_data['durations'] = np.concatenate(bootstrap_data['durations'], axis=0)
bootstrap_data['events']    = np.concatenate(bootstrap_data['events'],    axis=0)
bootstrap_data['a_class']   = np.concatenate(bootstrap_data['a_class'],   axis=0)
bootstrap_data['g_class']   = np.concatenate(bootstrap_data['g_class'],   axis=0)

Loading /mnt/d/pydatascience/g3_regress/data/results/bootstrap_predictions/20250427_train/keys_001/bootstrap_iteration_1.h5...
Loaded bootstrap iterations from /mnt/d/pydatascience/g3_regress/data/results/bootstrap_predictions/20250427_train/keys_001.
Loading /mnt/d/pydatascience/g3_regress/data/results/bootstrap_predictions/20250427_train/keys_002/bootstrap_iteration_1.h5...
Loaded bootstrap iterations from /mnt/d/pydatascience/g3_regress/data/results/bootstrap_predictions/20250427_train/keys_002.
Loading /mnt/d/pydatascience/g3_regress/data/results/bootstrap_predictions/20250427_train/keys_003/bootstrap_iteration_1.h5...
Loaded bootstrap iterations from /mnt/d/pydatascience/g3_regress/data/results/bootstrap_predictions/20250427_train/keys_003.
Loading /mnt/d/pydatascience/g3_regress/data/results/bootstrap_predictions/20250427_train/keys_004/bootstrap_iteration_1.h5...
Loaded bootstrap iterations from /mnt/d/pydatascience/g3_regress/data/results/bootstrap_predictions/20250427_train/ke

## 9. Create result prediction dataframe for training set

In [192]:
data = {
    'event': [],
    'Time Point': [],
    'A Class': [],
    'G Class': [],
    'Predicted': [],
    'Observed': []
}

model_preds = bootstrap_data['predictions']
combo = tuple(model_preds.keys())
cif_arrays = { m: model_preds[m] for m in combo }
ensembled_cif = ensemble_cif_arrays(cif_arrays)

durations = bootstrap_data['durations']   # shape (n_samples,)
events    = bootstrap_data['events']      # shape (n_samples,)
a_class   = bootstrap_data['a_class']     # shape (n_samples,)
g_class   = bootstrap_data['g_class']     # shape (n_samples,)

for e in range(ensembled_cif.shape[0]):
    for t in range(1, ensembled_cif.shape[1]):  # time points 1y to 5y
        probs = ensembled_cif[e][t]
        for ac in np.unique(a_class):
            for gc in np.unique(g_class):
                mask = (a_class == ac) & (g_class == gc)
                if not np.any(mask):
                    continue

                kmf = KaplanMeierFitter()
                kmf.fit(durations[mask], events[mask] == (e + 1))

                if t not in kmf.cumulative_density_.index:
                    closest = kmf.cumulative_density_.index.get_indexer([t], method='nearest')[0]
                    obs = kmf.cumulative_density_.iloc[closest].values[0]
                else:
                    obs = kmf.cumulative_density_.loc[t].values[0]

                pred = np.mean(probs[mask])

                data['event'].append(e + 1)
                data['Time Point'].append(t)
                data['A Class'].append(ac)
                data['G Class'].append(gc)
                data['Predicted'].append(pred)
                data['Observed'].append(obs)

result_df = pd.DataFrame(data)

## 10. Plot observed risk of Event 1 and 2 from training set

In [193]:
plot_observed_events(result_df, 
                     save_path='/mnt/d/pydatascience/g3_regress/doc/train_observed_events_by_ag_stage.png'
                    )


## 11. Plot predict and observe risk calibration of training set

In [194]:
plot_event_risk(
    result_df,
    event_num=1,
    save_path='/mnt/d/pydatascience/g3_regress/doc/train_event_1_by_ag_stage.png'
)

In [195]:
plot_event_risk(
    result_df,
    event_num=2,
    save_path='/mnt/d/pydatascience/g3_regress/doc/train_event_2_by_ag_stage.png'
)